# Geocoding Experiments: Multi-Strategy Evaluation

This notebook evaluates the **geolocation accuracy** of T5 model location extraction predictions using three different geocoding strategies:

**Dataset Filtering:**
- Only incidents where the model extracted village and/or other_locations info (village-level predictions)

**Three-Way Comparison:**
- Model predictions (T5 extracted locations)
- Human annotations (structured annotations)
- Manual baseline (human-looked-up coordinates from Google Maps)

**Three Geocoding Strategies:**
1. **Strategy 1 - Full Component Filtering**: Uses all fields (state, district, village, other_locations) with Google Maps component filtering
2. **Strategy 2 - Village Only**: Uses only village field (excluding other_locations) to test if messy location strings hurt accuracy
3. **Strategy 3 - Comma-Delimited**: Simple comma-separated list without component filtering to test Google's natural language processing

**Key Questions:**
- How does model geocoding accuracy compare to human annotations?
- Does including other_locations improve or hurt geocoding accuracy?
- Is component filtering better than simple comma-delimited queries?

## 1. Setup and Configuration

### 1.1 Colab Setup

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"  # Change if working on a different branch

# Clone fresh each session 
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install required packages
!pip install -qU pip setuptools wheel
!pip install --upgrade transformers datasets evaluate rouge-score rapidfuzz

# 4) Make result directories
import pathlib
import sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/location-extraction").mkdir(parents=True, exist_ok=True)

# 5) Import utilities from cloned repo (using importlib to handle hyphens in path)
try:
    import importlib.util
    
    spec = importlib.util.spec_from_file_location(
        "file_io",
        "/content/code-satp/models/classification-models/utils/file_io.py"
    )
    file_io = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(file_io)
    
    get_task_results_dir = file_io.get_task_results_dir
    save_dataframe_csv = file_io.save_dataframe_csv
    
    # Define task name for consistent file organization
    TASK_NAME = "geocoding"
    
    print("="*80)
    print("✅ SETUP COMPLETE")
    print("="*80)
    print(f"📂 Cloned repo: /content/code-satp")
    print(f"📂 Results dir: {get_task_results_dir(TASK_NAME)}")
    print(f"✅ File I/O utilities loaded successfully")
    print("="*80)
    
except Exception as e:
    print("="*80)
    print("⚠️  WARNING: Could not load file_io utilities")
    print(f"Error: {e}")
    print("="*80)
    print("Falling back to local mode - files will be saved to current directory")
    
    # Define fallback functions
    TASK_NAME = "location-extraction"
    
    def get_task_results_dir(task_name):
        return pathlib.Path(f"./results/{task_name}")
    
    def save_dataframe_csv(df, filename, task_name=None):
        output_path = f"./results/{task_name}/{filename}" if task_name else f"./{filename}"
        pathlib.Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)
        return output_path
    
    print("="*80)

### 1.2 Import Libraries

In [ ]:
# Core libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
from math import radians, sin, cos, sqrt, atan2

# API and HTTP
import requests

# Visualization
import matplotlib.pyplot as plt

# Colab-specific (will fail gracefully on local)
try:
    from google.colab import userdata
except ImportError:
    userdata = None
    print('⚠️  Not running on Colab - API key should be set manually')


## 2. Helper Functions

### 2.1 Parse Structured Location Output


In [ ]:
def parse_structured_location(text):
    """Parse structured location string into a dictionary."""
    location_dict = {
        'state': None,
        'district': None,
        'village': None,
        'other_locations': None
    }
    
    if not text or text.strip() == '':
        return location_dict
    
    # Split by comma and parse each part
    parts = [part.strip() for part in text.split(',')]
    for part in parts:
        if ':' in part:
            label, value = part.split(':', 1)
            label = label.strip().lower()
            value = value.strip()
            if label in location_dict:
                location_dict[label] = value
    
    return location_dict

# Test the parser with a sample
sample_location = "state: Chhattisgarh, district: Sukma, village: Nilamadgu, other_locations: Dornapal"
parsed = parse_structured_location(sample_location)
print("Sample parsing test:")
print(f"Input: {sample_location}")
print(f"Parsed: {parsed}")

### 2.2 Check for Village/Other Locations Info

In [ ]:
def has_village_or_other_locations(structured_location):
    """
    Check if a structured location has village and/or other_locations info.
    
    Args:
        structured_location: Structured location string
        
    Returns:
        Boolean: True if village or other_locations field has a value
    """
    if not structured_location or structured_location.strip() == '':
        return False
    
    parsed = parse_structured_location(structured_location)
    return bool(parsed['village'] or parsed['other_locations'])

print("✅ Village/other locations check function defined")


### 2.3 Distance Calculation

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points on Earth (in km).
    """
    if any(x is None for x in [lat1, lon1, lat2, lon2]):
        return None
    
    # Convert to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    # Earth's radius in kilometers
    R = 6371.0
    
    return R * c

print("✅ Distance calculation function defined")

In [ ]:
# Helper functions defined above - ready to proceed with data preparation

## 3. Data Preparation and Geocoding Setup

This section:
1. Loads and configures Google Maps API
2. Defines geocoding functions with strategy support
3. Loads predictions and merges with manual baseline coordinates
4. Filters to village-level predictions only

### 3.1 Setup: Load Libraries and API Key

In [ ]:
import requests
import pandas as pd

# Google Maps API key
# For Colab: stored securely in Colab Secrets
#   Add your key: Click 🔑 icon → Add secret → Name: 'googlemapsAPI' → Paste key → Enable notebook access
# For local: set as environment variable or replace with your key directly
try:
    from google.colab import userdata
    API_KEY = userdata.get('googlemapsAPI')
except ImportError:
    # Local execution - get from environment variable or set directly
    import os
    API_KEY = os.environ.get('GOOGLE_MAPS_API_KEY', None)
    if not API_KEY:
        print("⚠️  No API key found. Set GOOGLE_MAPS_API_KEY environment variable or add API_KEY here")
        # Uncomment and paste your API key for local execution:
        # API_KEY = "YOUR_API_KEY_HERE"

GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"

print("✅ Google Maps API configured")
print(f"   API key loaded: {API_KEY[:10]}..." if API_KEY else "⚠️  No API key found - configure API key above")

Using device: cuda
Using device: cuda


### 3.2 Load and Filter Dataset

Load predictions, merge with manual baseline coordinates, and filter to only incidents where the model extracted village and/or other_locations info.

In [ ]:
# Load test predictions (prefer Flan‑T5‑large file)
try:
    test_predictions = pd.read_csv(
        "/Users/ejt/Library/CloudStorage/Dropbox/projects/satp/code-satp/papers/location-extraction/results/location-extraction-seq2seq/location_flan-t5-large_predictions.csv"
    )
    print("✅ Loaded Flan‑T5‑large predictions")
except FileNotFoundError:
    # Original fallback(s)
    try:
        results_dir = get_task_results_dir(TASK_NAME)
        predictions_path = f"{results_dir}/location_extraction_test_predictions.csv"
        test_predictions = pd.read_csv(predictions_path)
        print(f"✅ Loaded test predictions from: {predictions_path}")
    except (NameError, FileNotFoundError):
        test_predictions = pd.read_csv("./location_extraction_test_predictions.csv")
        print("✅ Loaded test predictions from current directory")

print(f"   Total incidents: {len(test_predictions)}")

# Column alignment for this notebook
test_predictions = test_predictions.rename(
    columns={
        "prediction": "model_prediction",
        "true_label": "ground_truth",
    }
)

# Load manual baseline coordinates
url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/location_info_augmented.csv"
baseline_data = pd.read_csv(url, dtype=str)

# Convert latitude/longitude to float
baseline_data['baseline_lat'] = pd.to_numeric(baseline_data['latitude'], errors='coerce')
baseline_data['baseline_lon'] = pd.to_numeric(baseline_data['longitude'], errors='coerce')

print(f"\n✅ Loaded baseline data: {len(baseline_data)} total incidents")
print(f"   Manual coordinates available: {baseline_data['baseline_lat'].notna().sum()} incidents")

# Merge with baseline coordinates using incident_number
test_predictions['incident_number'] = test_predictions['incident_number'].astype(str)
baseline_data['incident_number'] = baseline_data['incident_number'].astype(str)

test_predictions_with_baseline = test_predictions.merge(
    baseline_data[['incident_number', 'baseline_lat', 'baseline_lon']],
    on='incident_number',
    how='left'
)

print(f"\n✅ Merged with baseline: {test_predictions_with_baseline['baseline_lat'].notna().sum()}/{len(test_predictions_with_baseline)} incidents have manual baseline coordinates")

# Filter to only incidents where MODEL extracted village and/or other_locations
print("\n" + "=" * 80)
print("FILTERING TO VILLAGE-LEVEL PREDICTIONS")
print("=" * 80)

test_predictions_with_baseline['has_village_info'] = test_predictions_with_baseline['model_prediction'].apply(
    has_village_or_other_locations
)

filtered_data = test_predictions_with_baseline[test_predictions_with_baseline['has_village_info']].copy()

print(f"\nFiltering based on MODEL predictions having village and/or other_locations:")
print(f"  Before filtering: {len(test_predictions_with_baseline)} incidents")
print(f"  After filtering:  {len(filtered_data)} incidents")
print(f"  Filtered out:     {len(test_predictions_with_baseline) - len(filtered_data)} incidents (district/state-only)")
print(f"  Retention rate:   {len(filtered_data)/len(test_predictions_with_baseline)*100:.1f}%")

print(f"\nFiltered dataset with baseline coordinates: {filtered_data['baseline_lat'].notna().sum()}/{len(filtered_data)} incidents")

print("\n" + "=" * 80)
print(f"✅ FINAL FILTERED DATASET: {len(filtered_data)} incidents")
print("=" * 80)

print("\nSample of filtered data:")
display(filtered_data[['model_prediction', 'ground_truth', 'baseline_lat', 'baseline_lon']].head(3))

### 3.3 Define Geocoding Functions

#### 3.3.1 Get Location Details

In [ ]:
def get_location_details(address, components=None):
    """
    Get location details using Google Geocoding API with optional component filtering.
    
    Args:
        address: Address string (village + other_locations)
        components: Optional dict with 'state' and/or 'district' for filtering
        
    Returns:
        Dict with state, district, subdistrict, town_village, latitude, longitude
    """
    # Build component filter string
    component_parts = ['country:IN']  # Always filter by India
    
    if components:
        if components.get('state'):
            component_parts.append(f"administrative_area_level_1:{components['state']}")
        if components.get('district'):
            component_parts.append(f"administrative_area_level_2:{components['district']}")
    
    params = {
        'address': address,
        'key': API_KEY,
        'components': '|'.join(component_parts)
    }
    
    response = requests.get(GEOCODE_URL, params=params)
    if response.status_code != 200:
        print(f"Error in API call: {response.status_code}")
        return None

    data = response.json()
    if data['status'] != 'OK':
        # Don't print error for ZERO_RESULTS (expected for some locations)
        if data['status'] != 'ZERO_RESULTS':
            print(f"Geocoding API error: {data['status']}")
        return None

    # Initialize components
    state = district = subdistrict = town_village = None
    latitude = longitude = None
    found_level = None

    # Iterate over results to find the most specific level
    for result in data.get('results', []):
        temp_state = temp_district = temp_subdistrict = temp_town_village = None
        address_components = result['address_components']

        # Map address components
        for component in address_components:
            types = component['types']
            if 'administrative_area_level_1' in types:
                temp_state = component['long_name']
            elif 'administrative_area_level_2' in types:
                temp_district = component['long_name']
            elif 'administrative_area_level_3' in types:
                temp_subdistrict = component['long_name']
            elif 'locality' in types:
                temp_town_village = component['long_name']
            elif 'sublocality' in types and not temp_town_village:
                temp_town_village = component['long_name']

        # Determine the most specific level in this result
        if temp_town_village and found_level not in ['town_village']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = temp_town_village
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'town_village'
        elif temp_subdistrict and found_level not in ['town_village', 'subdistrict']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'subdistrict'
        elif temp_district and found_level not in ['town_village', 'subdistrict', 'district']:
            state = temp_state
            district = temp_district
            subdistrict = None
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'district'

        # Break the loop if the most specific level is found
        if found_level == 'town_village':
            break

    return {
        'state': state,
        'district': district,
        'subdistrict': subdistrict,
        'town_village': town_village,
        'latitude': latitude,
        'longitude': longitude
    }

print("✅ Geocoding function defined")

#### 3.3.2 Parse Structured to Address and Components

In [ ]:
def parse_structured_to_address_and_components(structured_location, strategy="full"):
    """
    Parse T5 structured output and build address + components for Google API.
    
    Args:
        structured_location: Structured location string
        strategy: Geocoding strategy to use
            - "full": Component filtering with all fields (state, district, village, other_locations)
            - "village_only": Component filtering using only village (exclude other_locations)
            - "comma_delimited": Simple comma-delimited list without component filtering
    
    Strategy with fallbacks:
    1. Prefer village + other_locations as address (most specific)
    2. Fall back to district if no village/other_locations (district centroid)
    3. Fall back to state if only state available (state centroid)
    4. Use appropriate component filters based on what's in the address and strategy
    
    Examples:
    Input: "state: Chhattisgarh, district: Sukma, village: Nilamadgu, other_locations: Dornapal"
    Output: {'address': 'Nilamadgu, Dornapal', 'components': {'state': 'Chhattisgarh', 'district': 'Sukma'}}
    
    Input: "state: Chhattisgarh, district: Sukma" (no village)
    Output: {'address': 'Sukma', 'components': {'state': 'Chhattisgarh'}}
    
    Input: "state: Chhattisgarh" (only state)
    Output: {'address': 'Chhattisgarh', 'components': None}
    """
    if not structured_location or structured_location.strip() == '':
        return None
    
    parsed = parse_structured_location(structured_location)
    
    # Handle comma_delimited strategy - simple comma list without component filtering
    if strategy == "comma_delimited":
        address_parts = []
        if parsed['state']:
            address_parts.append(parsed['state'])
        if parsed['district']:
            address_parts.append(parsed['district'])
        if parsed['village']:
            address_parts.append(parsed['village'])
        if parsed['other_locations']:
            address_parts.append(parsed['other_locations'])
        
        if not address_parts:
            return None
        
        return {
            'address': ', '.join(address_parts),
            'components': None  # No component filtering for comma_delimited
        }
    
    # Handle village_only and full strategies with component filtering
    # Build address from most specific to least specific
    address_parts = []
    
    # Level 1: Village (and other_locations for "full" strategy only)
    if parsed['village']:
        address_parts.append(parsed['village'])
    if strategy == "full" and parsed['other_locations']:
        address_parts.append(parsed['other_locations'])
    
    # Level 2: Fallback to district if no village/other_locations
    if not address_parts and parsed['district']:
        address_parts.append(parsed['district'])
    
    # Level 3: Fallback to state if nothing else available
    if not address_parts and parsed['state']:
        address_parts.append(parsed['state'])
    
    # Build component filters (don't filter by what's already in the address)
    components = {}
    
    # If we're using village/other_locations as address, use state+district as filters
    if parsed['village'] or (strategy == "full" and parsed['other_locations']):
        if parsed['state']:
            components['state'] = parsed['state']
        if parsed['district']:
            components['district'] = parsed['district']
    
    # If we're using district as address, only use state as filter
    elif parsed['district'] and address_parts and parsed['district'] in address_parts[0]:
        if parsed['state']:
            components['state'] = parsed['state']
    
    # If we're using state as address, no additional filters needed
    # (Google will understand "Chhattisgarh, India" without extra components)
    
    # If we have nothing to geocode, return None
    if not address_parts:
        return None
    
    return {
        'address': ', '.join(address_parts),
        'components': components if components else None
    }

print("✅ Parser function defined with 3 strategies:")
print("   - full: Component filtering with all fields")
print("   - village_only: Component filtering with village only (no other_locations)")
print("   - comma_delimited: Simple comma list without component filtering")

#### 3.3.3 Unified Geocoding Function

This function geocodes all three sources (model predictions, human annotations, and manual baseline) for a given incident using a specified strategy.


In [ ]:
def geocode_incident_all_sources(row, strategy="full"):
    """
    Geocode an incident using all three sources: model predictions, human annotations, and manual baseline.
    
    Args:
        row: DataFrame row containing:
            - 'model_prediction': T5 model extracted location
            - 'ground_truth': Human annotation location
            - 'baseline_lat', 'baseline_lon': Manual baseline coordinates
            - Other incident metadata
        strategy: Geocoding strategy ("full", "village_only", or "comma_delimited")
    
    Returns:
        Dictionary with geocoded coordinates and distances for all three sources
    """
    result = {
        'incident_number': row['incident_number'],
        'incident_summary': row['incident_summary'],
        'model_prediction': row['model_prediction'],
        'ground_truth': row['ground_truth'],
        'strategy': strategy
    }
    
    # Geocode model prediction
    parsed_model = parse_structured_to_address_and_components(row['model_prediction'], strategy=strategy)
    if parsed_model:
        location_details = get_location_details(parsed_model['address'], parsed_model['components'])
        if location_details:
            result['model_lat'] = location_details['latitude']
            result['model_lon'] = location_details['longitude']
            result['model_geocoded_state'] = location_details['state']
            result['model_geocoded_district'] = location_details['district']
        else:
            result['model_lat'] = None
            result['model_lon'] = None
            result['model_geocoded_state'] = None
            result['model_geocoded_district'] = None
    else:
        result['model_lat'] = None
        result['model_lon'] = None
        result['model_geocoded_state'] = None
        result['model_geocoded_district'] = None
    
    # Geocode human annotation (ground truth)
    parsed_human = parse_structured_to_address_and_components(row['ground_truth'], strategy=strategy)
    if parsed_human:
        location_details = get_location_details(parsed_human['address'], parsed_human['components'])
        if location_details:
            result['human_lat'] = location_details['latitude']
            result['human_lon'] = location_details['longitude']
            result['human_geocoded_state'] = location_details['state']
            result['human_geocoded_district'] = location_details['district']
        else:
            result['human_lat'] = None
            result['human_lon'] = None
            result['human_geocoded_state'] = None
            result['human_geocoded_district'] = None
    else:
        result['human_lat'] = None
        result['human_lon'] = None
        result['human_geocoded_state'] = None
        result['human_geocoded_district'] = None
    
    # Manual baseline (already geocoded - just copy)
    result['baseline_lat'] = row.get('baseline_lat', None)
    result['baseline_lon'] = row.get('baseline_lon', None)
    
    # Calculate distances
    # Model → Baseline
    result['dist_model_baseline_km'] = haversine_distance(
        result['model_lat'], result['model_lon'],
        result['baseline_lat'], result['baseline_lon']
    )
    
    # Human → Baseline
    result['dist_human_baseline_km'] = haversine_distance(
        result['human_lat'], result['human_lon'],
        result['baseline_lat'], result['baseline_lon']
    )
    
    # Model → Human
    result['dist_model_human_km'] = haversine_distance(
        result['model_lat'], result['model_lon'],
        result['human_lat'], result['human_lon']
    )
    
    return result

print("✅ Unified geocoding function defined")
print("   Geocodes model, human, and baseline for any strategy")


## 4. Strategy 1: Component Filtering with All Fields

Geocode using component filtering with state, district, village, AND other_locations. This is the most comprehensive approach.


In [ ]:
print("=" * 80)
print("STRATEGY 1: COMPONENT FILTERING WITH ALL FIELDS")
print("=" * 80)
print(f"\nGeocoding {len(filtered_data)} incidents...")
print("Using: state, district, village, other_locations with component filtering")
print(f"Expected API calls: ~{len(filtered_data) * 2} (model + human for each incident)")
print("=" * 80)

strategy1_results = []

for idx, row in filtered_data.iterrows():
    result = geocode_incident_all_sources(row, strategy="full")
    strategy1_results.append(result)
    
    # Progress indicator
    if (len(strategy1_results)) % 10 == 0:
        print(f"Processed {len(strategy1_results)}/{len(filtered_data)} incidents...")

strategy1_df = pd.DataFrame(strategy1_results)

print(f"\n✅ Strategy 1 geocoding complete!")
print(f"   Model geocoded: {strategy1_df['model_lat'].notna().sum()}/{len(strategy1_df)}")
print(f"   Human geocoded: {strategy1_df['human_lat'].notna().sum()}/{len(strategy1_df)}")
print(f"   Baseline available: {strategy1_df['baseline_lat'].notna().sum()}/{len(strategy1_df)}")

# Calculate valid comparison counts
valid_model_baseline = (strategy1_df['dist_model_baseline_km'].notna()).sum()
valid_human_baseline = (strategy1_df['dist_human_baseline_km'].notna()).sum()
valid_model_human = (strategy1_df['dist_model_human_km'].notna()).sum()

print(f"\nValid comparisons:")
print(f"   Model → Baseline: {valid_model_baseline}")
print(f"   Human → Baseline: {valid_human_baseline}")
print(f"   Model → Human: {valid_model_human}")

# Display sample
print(f"\nSample results:")
display(strategy1_df[['model_prediction', 'model_lat', 'model_lon', 'human_lat', 'human_lon', 
                      'baseline_lat', 'baseline_lon', 'dist_model_baseline_km']].head(5))


### 4.1 Strategy 1: Distance Statistics


In [ ]:
print("=" * 80)
print("STRATEGY 1: DISTANCE STATISTICS")
print("=" * 80)

# Model → Baseline
valid_mb = strategy1_df[strategy1_df['dist_model_baseline_km'].notna()]
if len(valid_mb) > 0:
    print(f"\n📊 MODEL → BASELINE (n={len(valid_mb)}):")
    print(f"   Mean:   {valid_mb['dist_model_baseline_km'].mean():.2f} km")
    print(f"   Median: {valid_mb['dist_model_baseline_km'].median():.2f} km")
    print(f"   Std:    {valid_mb['dist_model_baseline_km'].std():.2f} km")
    print(f"   Min:    {valid_mb['dist_model_baseline_km'].min():.2f} km")
    print(f"   Max:    {valid_mb['dist_model_baseline_km'].max():.2f} km")
    print(f"\n   Within 10 km:  {(valid_mb['dist_model_baseline_km'] <= 10).sum()} ({(valid_mb['dist_model_baseline_km'] <= 10).sum()/len(valid_mb)*100:.1f}%)")
    print(f"   Within 50 km:  {(valid_mb['dist_model_baseline_km'] <= 50).sum()} ({(valid_mb['dist_model_baseline_km'] <= 50).sum()/len(valid_mb)*100:.1f}%)")
    print(f"   Within 100 km: {(valid_mb['dist_model_baseline_km'] <= 100).sum()} ({(valid_mb['dist_model_baseline_km'] <= 100).sum()/len(valid_mb)*100:.1f}%)")

# Human → Baseline
valid_hb = strategy1_df[strategy1_df['dist_human_baseline_km'].notna()]
if len(valid_hb) > 0:
    print(f"\n📊 HUMAN → BASELINE (n={len(valid_hb)}):")
    print(f"   Mean:   {valid_hb['dist_human_baseline_km'].mean():.2f} km")
    print(f"   Median: {valid_hb['dist_human_baseline_km'].median():.2f} km")
    print(f"   Std:    {valid_hb['dist_human_baseline_km'].std():.2f} km")
    print(f"   Min:    {valid_hb['dist_human_baseline_km'].min():.2f} km")
    print(f"   Max:    {valid_hb['dist_human_baseline_km'].max():.2f} km")
    print(f"\n   Within 10 km:  {(valid_hb['dist_human_baseline_km'] <= 10).sum()} ({(valid_hb['dist_human_baseline_km'] <= 10).sum()/len(valid_hb)*100:.1f}%)")
    print(f"   Within 50 km:  {(valid_hb['dist_human_baseline_km'] <= 50).sum()} ({(valid_hb['dist_human_baseline_km'] <= 50).sum()/len(valid_hb)*100:.1f}%)")
    print(f"   Within 100 km: {(valid_hb['dist_human_baseline_km'] <= 100).sum()} ({(valid_hb['dist_human_baseline_km'] <= 100).sum()/len(valid_hb)*100:.1f}%)")

# Model → Human
valid_mh = strategy1_df[strategy1_df['dist_model_human_km'].notna()]
if len(valid_mh) > 0:
    print(f"\n📊 MODEL → HUMAN (n={len(valid_mh)}):")
    print(f"   Mean:   {valid_mh['dist_model_human_km'].mean():.2f} km")
    print(f"   Median: {valid_mh['dist_model_human_km'].median():.2f} km")
    print(f"   Std:    {valid_mh['dist_model_human_km'].std():.2f} km")
    print(f"   Min:    {valid_mh['dist_model_human_km'].min():.2f} km")
    print(f"   Max:    {valid_mh['dist_model_human_km'].max():.2f} km")
    print(f"\n   Within 10 km:  {(valid_mh['dist_model_human_km'] <= 10).sum()} ({(valid_mh['dist_model_human_km'] <= 10).sum()/len(valid_mh)*100:.1f}%)")
    print(f"   Within 50 km:  {(valid_mh['dist_model_human_km'] <= 50).sum()} ({(valid_mh['dist_model_human_km'] <= 50).sum()/len(valid_mh)*100:.1f}%)")
    print(f"   Within 100 km: {(valid_mh['dist_model_human_km'] <= 100).sum()} ({(valid_mh['dist_model_human_km'] <= 100).sum()/len(valid_mh)*100:.1f}%)")

print("\n" + "=" * 80)


### 4.2 Save Strategy 1 Results


In [ ]:
# Save Strategy 1 results
try:
    saved_path = save_dataframe_csv(strategy1_df, "geocoding_strategy1_full_components.csv", task_name=TASK_NAME)
    print(f"✅ Strategy 1 results saved to: {saved_path}")
except NameError:
    output_path = "./geocoding_strategy1_full_components.csv"
    strategy1_df.to_csv(output_path, index=False)
    print(f"✅ Strategy 1 results saved to: {output_path}")


## 5. Strategy 2: Component Filtering with Village Only

Geocode using component filtering with state, district, and village ONLY (excluding other_locations). This tests whether the other_locations field helps or hurts accuracy.


In [ ]:
print("=" * 80)
print("STRATEGY 2: COMPONENT FILTERING WITH VILLAGE ONLY")
print("=" * 80)
print(f"\nGeocoding {len(filtered_data)} incidents...")
print("Using: state, district, village ONLY (no other_locations) with component filtering")
print(f"Expected API calls: ~{len(filtered_data) * 2}")
print("=" * 80)

strategy2_results = []

for idx, row in filtered_data.iterrows():
    result = geocode_incident_all_sources(row, strategy="village_only")
    strategy2_results.append(result)
    
    if (len(strategy2_results)) % 10 == 0:
        print(f"Processed {len(strategy2_results)}/{len(filtered_data)} incidents...")

strategy2_df = pd.DataFrame(strategy2_results)

print(f"\n✅ Strategy 2 geocoding complete!")
print(f"   Model geocoded: {strategy2_df['model_lat'].notna().sum()}/{len(strategy2_df)}")
print(f"   Human geocoded: {strategy2_df['human_lat'].notna().sum()}/{len(strategy2_df)}")
print(f"   Baseline available: {strategy2_df['baseline_lat'].notna().sum()}/{len(strategy2_df)}")

valid_model_baseline_s2 = (strategy2_df['dist_model_baseline_km'].notna()).sum()
valid_human_baseline_s2 = (strategy2_df['dist_human_baseline_km'].notna()).sum()
valid_model_human_s2 = (strategy2_df['dist_model_human_km'].notna()).sum()

print(f"\nValid comparisons:")
print(f"   Model → Baseline: {valid_model_baseline_s2}")
print(f"   Human → Baseline: {valid_human_baseline_s2}")
print(f"   Model → Human: {valid_model_human_s2}")

# Statistics
print("\n" + "=" * 80)
print("STRATEGY 2: DISTANCE STATISTICS")
print("=" * 80)

valid_mb_s2 = strategy2_df[strategy2_df['dist_model_baseline_km'].notna()]
if len(valid_mb_s2) > 0:
    print(f"\n📊 MODEL → BASELINE (n={len(valid_mb_s2)}):")
    print(f"   Mean: {valid_mb_s2['dist_model_baseline_km'].mean():.2f} km | Median: {valid_mb_s2['dist_model_baseline_km'].median():.2f} km")
    print(f"   Within 10 km: {(valid_mb_s2['dist_model_baseline_km'] <= 10).sum()} ({(valid_mb_s2['dist_model_baseline_km'] <= 10).sum()/len(valid_mb_s2)*100:.1f}%)")
    print(f"   Within 50 km: {(valid_mb_s2['dist_model_baseline_km'] <= 50).sum()} ({(valid_mb_s2['dist_model_baseline_km'] <= 50).sum()/len(valid_mb_s2)*100:.1f}%)")

valid_hb_s2 = strategy2_df[strategy2_df['dist_human_baseline_km'].notna()]
if len(valid_hb_s2) > 0:
    print(f"\n📊 HUMAN → BASELINE (n={len(valid_hb_s2)}):")
    print(f"   Mean: {valid_hb_s2['dist_human_baseline_km'].mean():.2f} km | Median: {valid_hb_s2['dist_human_baseline_km'].median():.2f} km")
    print(f"   Within 10 km: {(valid_hb_s2['dist_human_baseline_km'] <= 10).sum()} ({(valid_hb_s2['dist_human_baseline_km'] <= 10).sum()/len(valid_hb_s2)*100:.1f}%)")
    print(f"   Within 50 km: {(valid_hb_s2['dist_human_baseline_km'] <= 50).sum()} ({(valid_hb_s2['dist_human_baseline_km'] <= 50).sum()/len(valid_hb_s2)*100:.1f}%)")

valid_mh_s2 = strategy2_df[strategy2_df['dist_model_human_km'].notna()]
if len(valid_mh_s2) > 0:
    print(f"\n📊 MODEL → HUMAN (n={len(valid_mh_s2)}):")
    print(f"   Mean: {valid_mh_s2['dist_model_human_km'].mean():.2f} km | Median: {valid_mh_s2['dist_model_human_km'].median():.2f} km")
    print(f"   Within 10 km: {(valid_mh_s2['dist_model_human_km'] <= 10).sum()} ({(valid_mh_s2['dist_model_human_km'] <= 10).sum()/len(valid_mh_s2)*100:.1f}%)")
    print(f"   Within 50 km: {(valid_mh_s2['dist_model_human_km'] <= 50).sum()} ({(valid_mh_s2['dist_model_human_km'] <= 50).sum()/len(valid_mh_s2)*100:.1f}%)")

print("\n" + "=" * 80)

# Save results
try:
    saved_path = save_dataframe_csv(strategy2_df, "geocoding_strategy2_village_only.csv", task_name=TASK_NAME)
    print(f"✅ Strategy 2 results saved to: {saved_path}")
except NameError:
    output_path = "./geocoding_strategy2_village_only.csv"
    strategy2_df.to_csv(output_path, index=False)
    print(f"✅ Strategy 2 results saved to: {output_path}")


## 6. Strategy 3: Comma-Delimited List (No Component Filtering)

Geocode using a simple comma-delimited list of all location fields without component filtering. This tests whether Google's natural language processing can handle unstructured location strings better than explicit component filtering.


In [ ]:
print("=" * 80)
print("STRATEGY 3: COMMA-DELIMITED LIST")
print("=" * 80)
print(f"\nGeocoding {len(filtered_data)} incidents...")
print("Using: Simple comma list (state, district, village, other_locations) without component filtering")
print(f"Expected API calls: ~{len(filtered_data) * 2}")
print("=" * 80)

strategy3_results = []

for idx, row in filtered_data.iterrows():
    result = geocode_incident_all_sources(row, strategy="comma_delimited")
    strategy3_results.append(result)
    
    if (len(strategy3_results)) % 10 == 0:
        print(f"Processed {len(strategy3_results)}/{len(filtered_data)} incidents...")

strategy3_df = pd.DataFrame(strategy3_results)

print(f"\n✅ Strategy 3 geocoding complete!")
print(f"   Model geocoded: {strategy3_df['model_lat'].notna().sum()}/{len(strategy3_df)}")
print(f"   Human geocoded: {strategy3_df['human_lat'].notna().sum()}/{len(strategy3_df)}")
print(f"   Baseline available: {strategy3_df['baseline_lat'].notna().sum()}/{len(strategy3_df)}")

valid_model_baseline_s3 = (strategy3_df['dist_model_baseline_km'].notna()).sum()
valid_human_baseline_s3 = (strategy3_df['dist_human_baseline_km'].notna()).sum()
valid_model_human_s3 = (strategy3_df['dist_model_human_km'].notna()).sum()

print(f"\nValid comparisons:")
print(f"   Model → Baseline: {valid_model_baseline_s3}")
print(f"   Human → Baseline: {valid_human_baseline_s3}")
print(f"   Model → Human: {valid_model_human_s3}")

# Statistics
print("\n" + "=" * 80)
print("STRATEGY 3: DISTANCE STATISTICS")
print("=" * 80)

valid_mb_s3 = strategy3_df[strategy3_df['dist_model_baseline_km'].notna()]
if len(valid_mb_s3) > 0:
    print(f"\n📊 MODEL → BASELINE (n={len(valid_mb_s3)}):")
    print(f"   Mean: {valid_mb_s3['dist_model_baseline_km'].mean():.2f} km | Median: {valid_mb_s3['dist_model_baseline_km'].median():.2f} km")
    print(f"   Within 10 km: {(valid_mb_s3['dist_model_baseline_km'] <= 10).sum()} ({(valid_mb_s3['dist_model_baseline_km'] <= 10).sum()/len(valid_mb_s3)*100:.1f}%)")
    print(f"   Within 50 km: {(valid_mb_s3['dist_model_baseline_km'] <= 50).sum()} ({(valid_mb_s3['dist_model_baseline_km'] <= 50).sum()/len(valid_mb_s3)*100:.1f}%)")

valid_hb_s3 = strategy3_df[strategy3_df['dist_human_baseline_km'].notna()]
if len(valid_hb_s3) > 0:
    print(f"\n📊 HUMAN → BASELINE (n={len(valid_hb_s3)}):")
    print(f"   Mean: {valid_hb_s3['dist_human_baseline_km'].mean():.2f} km | Median: {valid_hb_s3['dist_human_baseline_km'].median():.2f} km")
    print(f"   Within 10 km: {(valid_hb_s3['dist_human_baseline_km'] <= 10).sum()} ({(valid_hb_s3['dist_human_baseline_km'] <= 10).sum()/len(valid_hb_s3)*100:.1f}%)")
    print(f"   Within 50 km: {(valid_hb_s3['dist_human_baseline_km'] <= 50).sum()} ({(valid_hb_s3['dist_human_baseline_km'] <= 50).sum()/len(valid_hb_s3)*100:.1f}%)")

valid_mh_s3 = strategy3_df[strategy3_df['dist_model_human_km'].notna()]
if len(valid_mh_s3) > 0:
    print(f"\n📊 MODEL → HUMAN (n={len(valid_mh_s3)}):")
    print(f"   Mean: {valid_mh_s3['dist_model_human_km'].mean():.2f} km | Median: {valid_mh_s3['dist_model_human_km'].median():.2f} km")
    print(f"   Within 10 km: {(valid_mh_s3['dist_model_human_km'] <= 10).sum()} ({(valid_mh_s3['dist_model_human_km'] <= 10).sum()/len(valid_mh_s3)*100:.1f}%)")
    print(f"   Within 50 km: {(valid_mh_s3['dist_model_human_km'] <= 50).sum()} ({(valid_mh_s3['dist_model_human_km'] <= 50).sum()/len(valid_mh_s3)*100:.1f}%)")

print("\n" + "=" * 80)

# Save results
try:
    saved_path = save_dataframe_csv(strategy3_df, "geocoding_strategy3_comma_delimited.csv", task_name=TASK_NAME)
    print(f"✅ Strategy 3 results saved to: {saved_path}")
except NameError:
    output_path = "./geocoding_strategy3_comma_delimited.csv"
    strategy3_df.to_csv(output_path, index=False)
    print(f"✅ Strategy 3 results saved to: {output_path}")


## 7. Cross-Strategy Comparison

Compare all three geocoding strategies to determine which performs best for each comparison type (model→baseline, human→baseline, model→human).


In [ ]:
print("=" * 80)
print("CROSS-STRATEGY COMPARISON")
print("=" * 80)

# Build summary comparison table
comparison_data = []

for strategy_name, df in [("Strategy 1: Full Components", strategy1_df), 
                           ("Strategy 2: Village Only", strategy2_df), 
                           ("Strategy 3: Comma-Delimited", strategy3_df)]:
    
    # Model → Baseline
    valid_mb = df[df['dist_model_baseline_km'].notna()]
    mb_mean = valid_mb['dist_model_baseline_km'].mean() if len(valid_mb) > 0 else None
    mb_median = valid_mb['dist_model_baseline_km'].median() if len(valid_mb) > 0 else None
    mb_within_10 = (valid_mb['dist_model_baseline_km'] <= 10).sum() if len(valid_mb) > 0 else 0
    mb_within_50 = (valid_mb['dist_model_baseline_km'] <= 50).sum() if len(valid_mb) > 0 else 0
    
    # Human → Baseline
    valid_hb = df[df['dist_human_baseline_km'].notna()]
    hb_mean = valid_hb['dist_human_baseline_km'].mean() if len(valid_hb) > 0 else None
    hb_median = valid_hb['dist_human_baseline_km'].median() if len(valid_hb) > 0 else None
    hb_within_10 = (valid_hb['dist_human_baseline_km'] <= 10).sum() if len(valid_hb) > 0 else 0
    hb_within_50 = (valid_hb['dist_human_baseline_km'] <= 50).sum() if len(valid_hb) > 0 else 0
    
    # Model → Human
    valid_mh = df[df['dist_model_human_km'].notna()]
    mh_mean = valid_mh['dist_model_human_km'].mean() if len(valid_mh) > 0 else None
    mh_median = valid_mh['dist_model_human_km'].median() if len(valid_mh) > 0 else None
    mh_within_10 = (valid_mh['dist_model_human_km'] <= 10).sum() if len(valid_mh) > 0 else 0
    mh_within_50 = (valid_mh['dist_model_human_km'] <= 50).sum() if len(valid_mh) > 0 else 0
    
    comparison_data.append({
        'Strategy': strategy_name,
        'Model→Baseline Mean (km)': mb_mean,
        'Model→Baseline Median (km)': mb_median,
        'Model→Baseline Within 10km': mb_within_10,
        'Model→Baseline Within 50km': mb_within_50,
        'Human→Baseline Mean (km)': hb_mean,
        'Human→Baseline Median (km)': hb_median,
        'Human→Baseline Within 10km': hb_within_10,
        'Human→Baseline Within 50km': hb_within_50,
        'Model→Human Mean (km)': mh_mean,
        'Model→Human Median (km)': mh_median,
        'Model→Human Within 10km': mh_within_10,
        'Model→Human Within 50km': mh_within_50,
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n📊 SUMMARY COMPARISON TABLE")
print("=" * 80)
display(comparison_df)

# Identify best strategies
print("\n" + "=" * 80)
print("BEST PERFORMING STRATEGIES")
print("=" * 80)

print(f"\n🏆 MODEL → BASELINE:")
print(f"   Lowest Mean Distance:   {comparison_df.loc[comparison_df['Model→Baseline Mean (km)'].idxmin(), 'Strategy']}")
print(f"   Lowest Median Distance: {comparison_df.loc[comparison_df['Model→Baseline Median (km)'].idxmin(), 'Strategy']}")
print(f"   Most Within 10km:       {comparison_df.loc[comparison_df['Model→Baseline Within 10km'].idxmax(), 'Strategy']}")

print(f"\n🏆 HUMAN → BASELINE:")
print(f"   Lowest Mean Distance:   {comparison_df.loc[comparison_df['Human→Baseline Mean (km)'].idxmin(), 'Strategy']}")
print(f"   Lowest Median Distance: {comparison_df.loc[comparison_df['Human→Baseline Median (km)'].idxmin(), 'Strategy']}")
print(f"   Most Within 10km:       {comparison_df.loc[comparison_df['Human→Baseline Within 10km'].idxmax(), 'Strategy']}")

print(f"\n🏆 MODEL → HUMAN:")
print(f"   Lowest Mean Distance:   {comparison_df.loc[comparison_df['Model→Human Mean (km)'].idxmin(), 'Strategy']}")
print(f"   Lowest Median Distance: {comparison_df.loc[comparison_df['Model→Human Median (km)'].idxmin(), 'Strategy']}")
print(f"   Most Within 10km:       {comparison_df.loc[comparison_df['Model→Human Within 10km'].idxmax(), 'Strategy']}")

print("\n" + "=" * 80)


### 7.1 Visualization: Strategy Comparison


In [ ]:
# Create side-by-side box plots comparing the three strategies
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

strategies = [
    ("Strategy 1: Full", strategy1_df),
    ("Strategy 2: Village Only", strategy2_df),
    ("Strategy 3: Comma-Delimited", strategy3_df)
]

# Model → Baseline
data_mb = [df[df['dist_model_baseline_km'].notna()]['dist_model_baseline_km'].values 
           for _, df in strategies]
bp1 = axes[0].boxplot(data_mb, labels=[s[0] for s in strategies], patch_artist=True)
axes[0].set_ylabel('Distance (km)')
axes[0].set_title('Model → Manual Baseline')
axes[0].set_xticklabels([s[0].replace(": ", ":\n") for s in strategies], fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')
for patch in bp1['boxes']:
    patch.set_facecolor('lightblue')

# Human → Baseline
data_hb = [df[df['dist_human_baseline_km'].notna()]['dist_human_baseline_km'].values 
           for _, df in strategies]
bp2 = axes[1].boxplot(data_hb, labels=[s[0] for s in strategies], patch_artist=True)
axes[1].set_ylabel('Distance (km)')
axes[1].set_title('Human → Manual Baseline')
axes[1].set_xticklabels([s[0].replace(": ", ":\n") for s in strategies], fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')
for patch in bp2['boxes']:
    patch.set_facecolor('lightgreen')

# Model → Human
data_mh = [df[df['dist_model_human_km'].notna()]['dist_model_human_km'].values 
           for _, df in strategies]
bp3 = axes[2].boxplot(data_mh, labels=[s[0] for s in strategies], patch_artist=True)
axes[2].set_ylabel('Distance (km)')
axes[2].set_title('Model → Human')
axes[2].set_xticklabels([s[0].replace(": ", ":\n") for s in strategies], fontsize=9)
axes[2].grid(True, alpha=0.3, axis='y')
for patch in bp3['boxes']:
    patch.set_facecolor('lightcoral')

plt.suptitle('Cross-Strategy Comparison: Distance Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Visualization complete")


### 7.2 Save Comparison Summary


In [ ]:
# Save cross-strategy comparison summary
try:
    saved_path = save_dataframe_csv(comparison_df, "geocoding_cross_strategy_comparison.csv", task_name=TASK_NAME)
    print(f"✅ Cross-strategy comparison saved to: {saved_path}")
except NameError:
    output_path = "./geocoding_cross_strategy_comparison.csv"
    comparison_df.to_csv(output_path, index=False)
    print(f"✅ Cross-strategy comparison saved to: {output_path}")

print("\n" + "=" * 80)
print("✅ GEOCODING EXPERIMENTS COMPLETE")
print("=" * 80)
print("\nAll three strategies have been evaluated:")
print("  • Strategy 1: Component filtering with all fields")
print("  • Strategy 2: Component filtering with village only")
print("  • Strategy 3: Comma-delimited list without component filtering")
print("\nResults saved for each strategy plus cross-strategy comparison.")
print("=" * 80)
